In [ ]:
"""
Semantic Segmentation with ResNet on Pascal VOC Dataset
Refined version with improved architecture, training practices, and evaluation metrics
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os
from typing import Tuple, Dict, List
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# =============================================================================
# 1. Configuration
# =============================================================================

class Config:
    """Configuration class for training parameters"""
    # Data parameters
    batch_size = 8
    num_workers = 4
    image_size = (256, 256)  # Slightly larger for better features
    
    # Model parameters
    num_classes = 21
    backbone = 'resnet50'  # Using ResNet50 for better features
    
    # Training parameters
    num_epochs = 30
    learning_rate = 1e-3
    weight_decay = 1e-4
    lr_schedule = 'cosine'  # 'step' or 'cosine'
    
    # Validation parameters
    val_interval = 1
    early_stopping_patience = 10
    
    # Paths
    data_root = './data'
    checkpoint_dir = './checkpoints'
    
    def __init__(self):
        os.makedirs(self.checkpoint_dir, exist_ok=True)

config = Config()

# =============================================================================
# 2. Enhanced Data Loading with Augmentation
# =============================================================================

class SegmentationAugmentation:
    """Custom augmentation for semantic segmentation"""
    
    def __init__(self, is_train=True, image_size=(256, 256)):
        self.is_train = is_train
        self.image_size = image_size
        
    def __call__(self, image, mask):
        # Resize
        image = transforms.functional.resize(image, self.image_size)
        mask = transforms.functional.resize(mask, self.image_size, interpolation=Image.NEAREST)
        
        if self.is_train:
            # Random horizontal flip
            if np.random.random() > 0.5:
                image = transforms.functional.hflip(image)
                mask = transforms.functional.hflip(mask)
            
            # Random color jitter
            color_jitter = transforms.ColorJitter(
                brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1
            )
            image = color_jitter(image)
            
            # Random rotation (-10 to 10 degrees)
            angle = np.random.uniform(-10, 10)
            image = transforms.functional.rotate(image, angle)
            mask = transforms.functional.rotate(mask, angle, fill=255)
        
        # Convert to tensor and normalize
        image = transforms.functional.to_tensor(image)
        image = transforms.functional.normalize(
            image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )
        
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        
        return image, mask

class VOCSegmentationDataset(Dataset):
    """Enhanced VOC Segmentation Dataset with augmentation"""
    
    def __init__(self, root, year='2012', image_set='train', download=True):
        self.dataset = torchvision.datasets.VOCSegmentation(
            root=root, year=year, image_set=image_set, download=download
        )
        self.augmentation = SegmentationAugmentation(
            is_train=(image_set == 'train'),
            image_size=config.image_size
        )
        
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        image, mask = self.dataset[idx]
        image, mask = self.augmentation(image, mask)
        return image, mask

# Load datasets
print("Loading Pascal VOC dataset...")
train_dataset = VOCSegmentationDataset(
    root=config.data_root, year='2012', image_set='train'
)
val_dataset = VOCSegmentationDataset(
    root=config.data_root, year='2012', image_set='val'
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

# Create data loaders
train_loader = DataLoader(
    train_dataset, batch_size=config.batch_size, 
    shuffle=True, num_workers=config.num_workers, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=config.batch_size, 
    shuffle=False, num_workers=config.num_workers, pin_memory=True
)

# =============================================================================
# 3. Improved Model Architecture
# =============================================================================

class ConvBlock(nn.Module):
    """Basic convolutional block with BN and ReLU"""
    
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class DecoderBlock(nn.Module):
    """Decoder block with skip connections"""
    
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.conv1 = ConvBlock(in_channels + skip_channels, out_channels)
        self.conv2 = ConvBlock(out_channels, out_channels)
        
    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x

class ImprovedResNetSegmentation(nn.Module):
    """Improved segmentation model with skip connections and better decoder"""
    
    def __init__(self, num_classes=21, backbone='resnet50'):
        super().__init__()
        
        # Load pretrained backbone
        if backbone == 'resnet50':
            resnet = torchvision.models.resnet50(pretrained=True)
            encoder_channels = [2048, 1024, 512, 256, 64]
        elif backbone == 'resnet34':
            resnet = torchvision.models.resnet34(pretrained=True)
            encoder_channels = [512, 256, 128, 64, 64]
        else:
            resnet = torchvision.models.resnet18(pretrained=True)
            encoder_channels = [512, 256, 128, 64, 64]
        
        # Encoder layers (extract intermediate features)
        self.encoder1 = nn.Sequential(*list(resnet.children())[:3])  # 64 channels
        self.encoder2 = nn.Sequential(*list(resnet.children())[3:5])  # 256/64 channels
        self.encoder3 = resnet.layer2  # 512/128 channels
        self.encoder4 = resnet.layer3  # 1024/256 channels
        self.encoder5 = resnet.layer4  # 2048/512 channels
        
        # Decoder with skip connections
        self.decoder4 = DecoderBlock(encoder_channels[0], encoder_channels[1], 512)
        self.decoder3 = DecoderBlock(512, encoder_channels[2], 256)
        self.decoder2 = DecoderBlock(256, encoder_channels[3], 128)
        self.decoder1 = DecoderBlock(128, encoder_channels[4], 64)
        
        # Final upsampling and classification
        self.final_conv = nn.Sequential(
            ConvBlock(64, 32),
            nn.Conv2d(32, num_classes, kernel_size=1)
        )
        
    def forward(self, x):
        # Encoder
        e1 = self.encoder1(x)  # 1/2
        e2 = self.encoder2(e1)  # 1/4
        e3 = self.encoder3(e2)  # 1/8
        e4 = self.encoder4(e3)  # 1/16
        e5 = self.encoder5(e4)  # 1/32
        
        # Decoder with skip connections
        d4 = self.decoder4(e5, e4)  # 1/16
        d3 = self.decoder3(d4, e3)  # 1/8
        d2 = self.decoder2(d3, e2)  # 1/4
        d1 = self.decoder1(d2, e1)  # 1/2
        
        # Final upsampling to original size
        out = self.final_conv(d1)
        out = F.interpolate(out, scale_factor=2, mode='bilinear', align_corners=False)
        
        return out

# Create model
model = ImprovedResNetSegmentation(
    num_classes=config.num_classes, 
    backbone=config.backbone
).to(device)

print(f"\nModel created with {config.backbone} backbone")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# =============================================================================
# 4. Metrics and Loss
# =============================================================================

class SegmentationMetrics:
    """Calculate IoU and other segmentation metrics"""
    
    def __init__(self, num_classes, ignore_index=255):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.reset()
        
    def reset(self):
        self.confusion_matrix = np.zeros((self.num_classes, self.num_classes))
        
    def update(self, predictions, targets):
        """Update confusion matrix with batch predictions"""
        valid_mask = targets != self.ignore_index
        predictions = predictions[valid_mask]
        targets = targets[valid_mask]
        
        for t, p in zip(targets.flatten(), predictions.flatten()):
            self.confusion_matrix[t, p] += 1
            
    def get_metrics(self):
        """Calculate IoU per class and mean IoU"""
        eps = 1e-6
        intersection = np.diag(self.confusion_matrix)
        union = (self.confusion_matrix.sum(axis=1) + 
                self.confusion_matrix.sum(axis=0) - intersection)
        
        iou_per_class = intersection / (union + eps)
        mean_iou = np.nanmean(iou_per_class)
        
        # Pixel accuracy
        pixel_acc = intersection.sum() / (self.confusion_matrix.sum() + eps)
        
        return {
            'mIoU': mean_iou,
            'IoU_per_class': iou_per_class,
            'pixel_accuracy': pixel_acc
        }

# Weighted loss for class imbalance
def calculate_class_weights(dataloader, num_classes):
    """Calculate class weights based on frequency"""
    class_counts = torch.zeros(num_classes)
    
    print("Calculating class weights...")
    for _, masks in dataloader:
        for c in range(num_classes):
            class_counts[c] += (masks == c).sum()
    
    # Inverse frequency weighting
    total_pixels = class_counts.sum()
    class_weights = total_pixels / (num_classes * class_counts + 1)
    class_weights = class_weights / class_weights.sum() * num_classes
    
    return class_weights.to(device)

# Calculate class weights (optional, can be slow)
# class_weights = calculate_class_weights(train_loader, config.num_classes)
class_weights = None  # Set to None to use uniform weights

criterion = nn.CrossEntropyLoss(
    weight=class_weights, 
    ignore_index=255
)

# =============================================================================
# 5. Training Functions
# =============================================================================

def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    metrics = SegmentationMetrics(config.num_classes)
    
    for i, (images, masks) in enumerate(dataloader):
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Update metrics
        predictions = torch.argmax(outputs, dim=1)
        metrics.update(
            predictions.cpu().numpy(), 
            masks.cpu().numpy()
        )
        running_loss += loss.item()
        
        # Print progress
        if i % 50 == 0:
            current_metrics = metrics.get_metrics()
            print(f'  Batch {i}/{len(dataloader)}, Loss: {loss.item():.4f}, '
                  f'mIoU: {current_metrics["mIoU"]:.4f}')
    
    epoch_metrics = metrics.get_metrics()
    epoch_metrics['loss'] = running_loss / len(dataloader)
    return epoch_metrics

def validate_epoch(model, dataloader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    metrics = SegmentationMetrics(config.num_classes)
    
    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            predictions = torch.argmax(outputs, dim=1)
            metrics.update(
                predictions.cpu().numpy(), 
                masks.cpu().numpy()
            )
            running_loss += loss.item()
    
    epoch_metrics = metrics.get_metrics()
    epoch_metrics['loss'] = running_loss / len(dataloader)
    return epoch_metrics

# =============================================================================
# 6. Training Loop with Early Stopping
# =============================================================================

# Optimizer and scheduler
optimizer = optim.AdamW(
    model.parameters(), 
    lr=config.learning_rate, 
    weight_decay=config.weight_decay
)

if config.lr_schedule == 'cosine':
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config.num_epochs
    )
else:
    scheduler = optim.lr_scheduler.StepLR(
        optimizer, step_size=10, gamma=0.1
    )

# Training history
history = {
    'train_loss': [], 'train_mIoU': [], 'train_acc': [],
    'val_loss': [], 'val_mIoU': [], 'val_acc': []
}

best_val_miou = 0
patience_counter = 0

print(f"\n{'='*60}")
print(f"Starting training for {config.num_epochs} epochs...")
print(f"{'='*60}")

for epoch in range(config.num_epochs):
    print(f'\nEpoch {epoch+1}/{config.num_epochs}')
    print('-' * 40)
    
    # Training
    train_metrics = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validation
    val_metrics = validate_epoch(model, val_loader, criterion, device)
    
    # Update learning rate
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Store history
    history['train_loss'].append(train_metrics['loss'])
    history['train_mIoU'].append(train_metrics['mIoU'])
    history['train_acc'].append(train_metrics['pixel_accuracy'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_mIoU'].append(val_metrics['mIoU'])
    history['val_acc'].append(val_metrics['pixel_accuracy'])
    
    # Print metrics
    print(f"Train - Loss: {train_metrics['loss']:.4f}, "
          f"mIoU: {train_metrics['mIoU']:.4f}, "
          f"Acc: {train_metrics['pixel_accuracy']:.4f}")
    print(f"Val   - Loss: {val_metrics['loss']:.4f}, "
          f"mIoU: {val_metrics['mIoU']:.4f}, "
          f"Acc: {val_metrics['pixel_accuracy']:.4f}")
    print(f"Learning Rate: {current_lr:.6f}")
    
    # Early stopping and model saving
    if val_metrics['mIoU'] > best_val_miou:
        best_val_miou = val_metrics['mIoU']
        patience_counter = 0
        
        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_miou': best_val_miou,
            'history': history
        }, os.path.join(config.checkpoint_dir, 'best_model.pth'))
        print(f"✓ Saved best model (mIoU: {best_val_miou:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= config.early_stopping_patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)
print(f"Best Validation mIoU: {best_val_miou:.4f}")
print(f"Final Train mIoU: {history['train_mIoU'][-1]:.4f}")
print(f"Final Val mIoU: {history['val_mIoU'][-1]:.4f}")

# =============================================================================
# 7. Visualization Functions
# =============================================================================

# Class names and colormap
class_names = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus',
    'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike',
    'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

def create_pascal_colormap():
    """Create Pascal VOC colormap"""
    def bit_get(val, idx):
        return (val >> idx) & 1
    
    colormap = np.zeros((256, 3), dtype=np.uint8)
    for i in range(256):
        r = g = b = 0
        for j in range(8):
            r |= bit_get(i, 0) << (7 - j)
            g |= bit_get(i, 1) << (7 - j)
            b |= bit_get(i, 2) << (7 - j)
            i >>= 3
        colormap[i] = [r, g, b]
    return colormap[:21]

colormap = create_pascal_colormap()

def plot_training_history(history):
    """Plot training curves"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss Curves')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # mIoU
    axes[1].plot(history['train_mIoU'], label='Train mIoU', linewidth=2)
    axes[1].plot(history['val_mIoU'], label='Val mIoU', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('mIoU')
    axes[1].set_title('Mean IoU Curves')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Accuracy
    axes[2].plot(history['train_acc'], label='Train Acc', linewidth=2)
    axes[2].plot(history['val_acc'], label='Val Acc', linewidth=2)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Pixel Accuracy')
    axes[2].set_title('Pixel Accuracy Curves')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def visualize_predictions(model, dataset, device, num_samples=3):
    """Visualize model predictions with overlay"""
    model.eval()
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    with torch.no_grad():
        for i in range(num_samples):
            image, mask = dataset[i]
            image_batch = image.unsqueeze(0).to(device)
            
            # Get prediction
            output = model(image_batch)
            prediction = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
            
            # Denormalize image
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            image_denorm = image * std + mean
            image_denorm = torch.clamp(image_denorm, 0, 1).permute(1, 2, 0).numpy()
            
            # Convert masks to color
            mask_np = mask.numpy()
            mask_colored = colormap[np.where(mask_np == 255, 0, mask_np)]
            pred_colored = colormap[prediction]
            
            # Create overlay
            overlay = image_denorm * 0.6 + pred_colored / 255.0 * 0.4
            
            # Plot
            axes[i, 0].imshow(image_denorm)
            axes[i, 0].set_title('Original Image')
            axes[i, 0].axis('off')
            
            axes[i, 1].imshow(mask_colored)
            axes[i, 1].set_title('Ground Truth')
            axes[i, 1].axis('off')
            
            axes[i, 2].imshow(pred_colored)
            axes[i, 2].set_title('Prediction')
            axes[i, 2].axis('off')
            
            axes[i, 3].imshow(overlay)
            axes[i, 3].set_title('Overlay')
            axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

# Plot results
print("\nPlotting training history...")
plot_training_history(history)

print("\nVisualizing predictions on validation samples...")
visualize_predictions(model, val_dataset, device, num_samples=3)

print("\n✓ Training pipeline completed successfully!")